# Etapa4 — Experimentos controlados de MLP

BOs de celulares subtraídos da capital. 0=FURTO, 1=ROUBO. Dados usados: 2023 e2024; 2025 reservado. Este notebook lê os treinamentos já executados e não dispara a bateria automaticamente. Baseline histórico e documentos anteriores preservados.

In [1]:
from pathlib import Path
import csv,json
ROOT=Path.cwd()
if not (ROOT/'reports').exists():ROOT=ROOT.parent
OUT=ROOT/'reports/etapa4'
def show(name,columns):
    with (OUT/(name+'.csv')).open(encoding='utf-8-sig') as f:
        for r in csv.DictReader(f):print({k:r[k] for k in columns})
print((OUT/'divisao_interna.json').read_text(encoding='utf-8'))


[
  {
    "particao": "treino_interno",
    "n": 144697,
    "mes_min": 1,
    "mes_max": 9,
    "furtos": 81128,
    "roubos": 63569,
    "keys_sha256": "bc0477356a5cea818e9675a34db5d14989dc5b354a5cf1837ba8082baf6ff2fb"
  },
  {
    "particao": "validacao_interna",
    "n": 45427,
    "mes_min": 10,
    "mes_max": 12,
    "furtos": 25293,
    "roubos": 20134,
    "keys_sha256": "f85ee407520d4456035fa208031980b527277fd33c4d3f7bc81cb941666277e8"
  }
]


## Metodologia

Jan–Set/2023 ajusta os transformadores e treina; Out–Dez/2023 orienta atributos, arquiteturas, regularização e épocas. Depois, reajuste completo de2023 com três épocas fixas e avaliação única em2024 por configuração/semente. Não há EarlyStopping em2024 nem calibrador. Meses Q4 não aparecem no treino interno; seus dummies não aprendem efeitos, limitação registrada. Matriz e protocolo foram salvos antes de treinar.

## Bloco A

A0 referência; A1 sem local; A2 sem bairro; A3 sem mês; A4 frequências10/100 contra50; A5 equivalência unívoca de acento de local. Não agrupa OUTROS com ausente. A5 reutiliza A0 após provar matrizes iguais. A3 ganhou pouco em Macro F1 e perdeu3,88pp de recall; a revisão qualitativa antes de B manteve os quatro atributos. Ver o desvio do desempate mecânico no protocolo.

In [2]:
show('comparacao_atributos',['id','input_dimension','retained_bairros','macro_f1','recall_roubo','delta_macro_f1_pp'])
print((OUT/'decisao_representacao.json').read_text(encoding='utf-8'))


{'id': 'A0', 'input_dimension': '267', 'retained_bairros': '215', 'macro_f1': '0.6808291307405228', 'recall_roubo': '0.645226979239098', 'delta_macro_f1_pp': '0.0'}
{'id': 'A1', 'input_dimension': '238', 'retained_bairros': '215', 'macro_f1': '0.6477646199294143', 'recall_roubo': '0.5734578325221019', 'delta_macro_f1_pp': '-3.3064510811108505'}
{'id': 'A2', 'input_dimension': '50', 'retained_bairros': '0', 'macro_f1': '0.5786361424736235', 'recall_roubo': '0.7258865600476806', 'delta_macro_f1_pp': '-10.219298826689926'}
{'id': 'A3', 'input_dimension': '254', 'retained_bairros': '215', 'macro_f1': '0.6821647478740092', 'recall_roubo': '0.6064368729512268', 'delta_macro_f1_pp': '0.13356171334863998'}
{'id': 'A4_100', 'input_dimension': '205', 'retained_bairros': '153', 'macro_f1': '0.6794270996543571', 'recall_roubo': '0.6342505215059104', 'delta_macro_f1_pp': '-0.1402031086165656'}
{'id': 'A4_10', 'input_dimension': '671', 'retained_bairros': '619', 'macro_f1': '0.6813945255308467', 're

![Atributos](../reports/etapa4/figuras/comparacao_atributos.png)

## Bloco B

Mesma representação A5: One-Hot dos quatro atributos e frequência50, sem variáveis proibidas. Redes32,64,64→32 comparadas com BCE, Adam0,001, batch256 e threshold0,5. Tabela abaixo é2024, semente42; as curvas com validação são internas de2023.

In [3]:
show('comparacao_arquiteturas',['id','parameters','epochs','accuracy','macro_f1','balanced_accuracy','precision_roubo','recall_roubo','precision_furto','recall_furto','brier','log_loss'])


{'id': 'B32', 'parameters': '9633', 'epochs': '3', 'accuracy': '0.6860064822681645', 'macro_f1': '0.677451837885561', 'balanced_accuracy': '0.6772168770679694', 'precision_roubo': '0.6280485962102209', 'recall_roubo': '0.6218282465589468', 'precision_furto': '0.727374699353948', 'recall_furto': '0.732605507576992', 'brier': '0.20140223315103842', 'log_loss': '0.5863302471642929'}
{'id': 'B64_32', 'parameters': '21313', 'epochs': '3', 'accuracy': '0.6810409389848642', 'macro_f1': '0.6751776702141328', 'balanced_accuracy': '0.6767630537496756', 'precision_roubo': '0.6142673285530429', 'recall_roubo': '0.6498055056852184', 'precision_furto': '0.7345784006894363', 'recall_furto': '0.7037206018141329', 'brier': '0.2027196048864563', 'log_loss': '0.588850584716284'}
{'id': 'B64', 'parameters': '19265', 'epochs': '3', 'accuracy': '0.6850687560967935', 'macro_f1': '0.676902006111324', 'balanced_accuracy': '0.6768849987913127', 'precision_roubo': '0.625754195798961', 'recall_roubo': '0.62531418

![Curvas](../reports/etapa4/figuras/curvas_arquiteturas.png)
![Comparação](../reports/etapa4/figuras/comparacao_arquiteturas.png)
![Matrizes](../reports/etapa4/figuras/matrizes_2024.png)

## Bloco C

Dropout0,1 e L2=0,0001 testados isoladamente; nenhum melhorou a Macro F1 interna. L2 reduz gap sem melhorar a BCE de validação. Não se acionou o gatilho prévio para LR menor. B64 e64→32 foram avaliadas com sementes42/7/123; a ordem dos lotes permaneceu RNG42.

In [4]:
show('comparacao_regularizacao',['id','macro_f1','recall_roubo','log_loss','gap_bce'])
show('resumo_validacao_2024',['id','n_seeds','macro_f1_mean','macro_f1_sd','recall_roubo_mean','macro_delta_historico_pp'])


{'id': 'B64', 'macro_f1': '0.6808291307405228', 'recall_roubo': '0.645226979239098', 'log_loss': '0.5870552784758206', 'gap_bce': '0.01064434240685952'}
{'id': 'C_dropout', 'macro_f1': '0.6806834144918027', 'recall_roubo': '0.6680242376080262', 'log_loss': '0.5890381244580806', 'gap_bce': '0.017378339893530437'}
{'id': 'C_l2', 'macro_f1': '0.6796415446787711', 'recall_roubo': '0.6660872156551108', 'log_loss': '0.5875692540975926', 'gap_bce': '0.008573045280406033'}
{'id': 'A0', 'n_seeds': '1', 'macro_f1_mean': '0.676902006111324', 'macro_f1_sd': '', 'recall_roubo_mean': '0.6253141831238779', 'macro_delta_historico_pp': '-0.0762097336954648'}
{'id': 'A1', 'n_seeds': '1', 'macro_f1_mean': '0.643565240944735', 'macro_f1_sd': '', 'recall_roubo_mean': '0.5616247755834829', 'macro_delta_historico_pp': '-3.409886250354366'}
{'id': 'B32', 'n_seeds': '1', 'macro_f1_mean': '0.677451837885561', 'macro_f1_sd': '', 'recall_roubo_mean': '0.6218282465589468', 'macro_delta_historico_pp': '-0.021226556

![Regularização](../reports/etapa4/figuras/regularizacao.png)
![Estabilidade](../reports/etapa4/figuras/estabilidade_sementes.png)

## Erros e interpretação

Mais profundidade aumentou recall de roubo e FP, sem ganho consistente de Macro F1. Baseline histórico usou EarlyStopping em2024 e época5; comparação nova usa épocas fixas escolhidas internamente. Pequenas diferenças não demonstram avanço comprovado. Ausências/raros/novos continuam problemáticos e as diferenças não são causalidade criminal ou ranking de segurança.

In [5]:
show('erros_vs_historico',['id','fp','fn','high_conf_errors','brier','log_loss'])
print((OUT/'testes.json').read_text(encoding='utf-8'))
print((OUT/'recomendacao_candidatos.json').read_text(encoding='utf-8'))


{'id': 'Historico', 'fp': '23727', 'fn': '25962', 'high_conf_errors': '879', 'brier': '0.2009063087680951', 'log_loss': '0.5846715435989691'}
{'id': 'A0', 'fp': '24997', 'fn': '25044', 'high_conf_errors': '609', 'brier': '0.20160448847627263', 'log_loss': '0.5865684835774249'}
{'id': 'A1', 'fp': '25438', 'fn': '29301', 'high_conf_errors': '49', 'brier': '0.21812154908789932', 'log_loss': '0.6265433226948016'}
{'id': 'B32', 'fp': '24615', 'fn': '25277', 'high_conf_errors': '536', 'brier': '0.20140223315103842', 'log_loss': '0.5863302471642929'}
{'id': 'B64_32', 'fp': '27274', 'fn': '23407', 'high_conf_errors': '608', 'brier': '0.2027196048864563', 'log_loss': '0.588850584716284'}
{'id': 'B64', 'fp': '24997', 'fn': '25044', 'high_conf_errors': '609', 'brier': '0.20160448847627263', 'log_loss': '0.5865684835774249'}
[
  {
    "arquivo": "test_integridade.py",
    "executados": 8,
    "falhas": 0,
    "erros": 0,
    "sucesso": true
  },
  {
    "arquivo": "test_ml_pipeline.py",
    "execu

## Limitações e Etapa5

Candidata principal experimental:64, com estabilidade em três sementes. A profunda é alternativa se priorizar recall; a compacta32 usa metade dos parâmetros e tem métrica próxima, mas apenas uma semente completa. Não há seleção definitiva nem calibração. Antes de2025, congelar arquitetura, orçamento, pré-processador, threshold e eventual protocolo de calibração. Ver `../docs/recomendacoes_etapa5.md` e `../reports/relatorio_etapa4.md`. Nenhuma aplicação web foi criada.